# 🎬 AI Service Backend - Google Colab Setup

Notebook này sẽ tự động hóa toàn bộ quá trình cài đặt và khởi chạy backend AI Service (bao gồm Wav2Lip, FFmpeg, ngrok) trên Google Colab với GPU miễn phí.

**Lưu ý trước khi chạy:**
1. Vào `Runtime` > `Change runtime type` > Chọn **T4 GPU**.
2. Đảm bảo bạn đã upload file checkpoint Wav2Lip (`Wav2Lip-SD-GAN.pt`) lên Google Drive của bạn (ví dụ: `MyDrive/[SEAHACKATHON]/Wav2Lip-SD-GAN.pt`).
3. Chạy lần lượt các cell bên dưới.

In [ ]:
# 1. Clone Source Code & Cài đặt Dependencies
import os
%cd /content

# Clone repo
!git clone https://github.com/sea-hackathon-2026/ai-service.git
%cd /content/ai-service

# Cài đặt FFmpeg
!apt-get update -y
!apt-get install -y ffmpeg

# Cài đặt thư viện Python cho AI Service
!pip install -r requirements.txt
!pip install fastapi uvicorn python-multipart pyngrok edge-tts pydantic librosa==0.9.2 nest_asyncio

In [ ]:
# 2. Cài đặt Wav2Lip
%cd /content
!git clone https://github.com/Rudrabha/Wav2Lip.git
%cd /content/Wav2Lip

# Cài requirements cho Wav2Lip
!pip install -r requirements.txt

# Tạo thư mục chứa model
!mkdir -p checkpoints
!mkdir -p face_detection/detection/sfd

# Tải model detect khuôn mặt (Face Detection)
!wget "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth" -O "face_detection/detection/sfd/s3fd.pth"

In [ ]:
# 3. Mount Google Drive & Copy Wav2Lip Checkpoint
from google.colab import drive
drive.mount('/content/drive')

# Thay đổi đường dẫn này nếu file checkpoint trên Drive của bạn nằm ở vị trí khác
DRIVE_CHECKPOINT_PATH = "/content/drive/MyDrive/[SEAHACKATHON]/Wav2Lip-SD-GAN.pt"
COLAB_CHECKPOINT_PATH = "/content/Wav2Lip/checkpoints/Wav2Lip-SD-GAN.pt"

!cp -f "$DRIVE_CHECKPOINT_PATH" "$COLAB_CHECKPOINT_PATH"
print("✅ Đã copy checkpoint Wav2Lip thành công!")

In [ ]:
# 4. Cấu hình Biến Môi Trường (API Keys)
import os

# 🔑 Điền Gemini API Key của bạn vào đây (để sinh kịch bản)
os.environ["GEMINI_API_KEY"] = ""

# 🔑 (Tùy chọn) Điền Ngrok Auth Token để Public URL không bị hết hạn hoặc thay đổi liên tục
# Lấy tại: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = ""

# Cấu hình hệ thống để dùng thư mục Wav2Lip trên Colab
os.environ["LIVESTREAM_ENABLE_WAV2LIP"] = "true"
os.environ["WAV2LIP_PATH"] = "/content/Wav2Lip"
os.environ["CHECKPOINT_PATH"] = "/content/Wav2Lip/checkpoints/Wav2Lip-SD-GAN.pt"

print("✅ Đã thiết lập biến môi trường.")

In [ ]:
# 5. Khởi chạy FastAPI Server với Ngrok
%cd /content/ai-service

import uvicorn
import nest_asyncio
from pyngrok import ngrok

# Giúp uvicorn chạy được bên trong Colab cell
nest_asyncio.apply()

# Xác thực Ngrok (nếu có)
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Mở port 8000 thông qua Ngrok
public_url = ngrok.connect(8000)
print("=" * 60)
print("🚀 PUBLIC URL (Dùng cho Frontend):", public_url)
print("📚 SWAGGER UI (Test API):", f"{public_url}/docs")
print("=" * 60)

# Chạy server (Colab cell này sẽ chạy liên tục để duy trì server)
uvicorn.run("app.main:app", host="0.0.0.0", port=8000)